In [1]:
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings('ignore')

check file is manually filled in, from the report and the TMT files, linking the tube code (e.g. M_H_20) to the corresponding label (amyloid-Beta Status), AIBL.Id, Collection

In [2]:
check_protemics = pd.read_excel('../raw_data/check.xlsx', sheet_name='proteomics')
print("Column included: ", check_protemics.columns)
print("Dimension: ", check_protemics.shape)

check_metabolomics = pd.read_excel('../raw_data/check.xlsx', sheet_name='metabolomics')
print("Column included: ", check_metabolomics.columns)
print("Dimension: ", check_metabolomics.shape)

check_amyloid = pd.read_excel('../raw_data/check.xlsx', sheet_name='amyloid')
print("Column included: ", check_amyloid.columns)
print("Dimension: ", check_amyloid.shape)

Column included:  Index(['label_name', 'Gender', 'label', 'AIBL.ID', 'Collection'], dtype='object')
Dimension:  (90, 5)
Column included:  Index(['label_name', 'Gender', 'label', 'AIBL.ID', 'Collection',
       'Tube Barcode'],
      dtype='object')
Dimension:  (90, 6)
Column included:  Index(['AIBL.ID', 'Gender', 'label', 'Image.PET.Centiloid'], dtype='object')
Dimension:  (90, 4)


#### For Proteomics

In [3]:
###### no sex covariant
proteomics_df = pd.read_csv('../raw_data/ORCA_IRS_COMBAT_nosexcovar_021024_Yihan.csv',index_col=0)
print(proteomics_df.shape)
## use the protein without imputations
proteomics_df = proteomics_df.transpose()
proteomics_df.reset_index(inplace=True)
proteomics_df.rename(columns={'index': 'AIBL.ID'}, inplace=True)
proteomics_df = proteomics_df.apply(lambda col: pd.to_numeric(col, errors='coerce') if col.name != 'label' else col)

(596, 90)


In [4]:
proteomics_cols = proteomics_df.columns.to_list()
proteomics_cols = [value for value in proteomics_cols if value not in ['AIBL.ID']]
print(proteomics_df.select_dtypes(include=['object', 'category']).columns)
proteomics_df = pd.merge(proteomics_df, check_protemics[['label','AIBL.ID', 'Collection','Gender']], on = 'AIBL.ID', how = 'left')
# proteomics_df.to_csv('../preprocessed_data/proteomics_with_label.csv')

Index([], dtype='object', name='name')


In [5]:
# proteomics_validation_df = pd.read_csv('../preprocessed_data/proteomics_validation.csv',index_col=0)
# proteomics_validation_df.reset_index(inplace=True)
# proteomics_validation_df = proteomics_validation_df.rename(columns={"condition": "label", "demographic.sex": "Gender", "image.pet.centiloid": "Centiloid"})
# proteomics_validation_cols = proteomics_validation_df.columns.to_list()
# proteomics_validation_cols = [value for value in proteomics_validation_cols if value not in ['AIBL.ID', 'label',"Gender", 'demographic.Gender',"Centiloid", "Collection"]]

#### For Metabolomics

In [6]:
metabolomics = pd.read_csv('../raw_data/Metabolomics_impute_Normaliseddata_noWave.csv',index_col=0)
metabolomics.reset_index(inplace=True)
metabolomics.rename(columns={'index': 'Tube Barcode'}, inplace=True)
metabolomics['Tube Barcode'] = metabolomics['Tube Barcode'].str.split('_').str[1]
metabolomics_merge_df = pd.merge(metabolomics, check_metabolomics, on = 'Tube Barcode', how = 'right')
metabolomics_merge_df = metabolomics_merge_df.drop(['Tube Barcode', 'Class', 'label_name'], axis=1)

In [7]:
metabolomics_cols = metabolomics_merge_df.columns.to_list()
metabolomics_cols = [value for value in metabolomics_cols if value not in ['AIBL.ID', 'Gender','label', 'Collection']]

In [8]:
# metabolomics_merge_df.to_csv('../preprocessed_data/metabolomics_with_label.csv')

#### Merge omics

In [9]:
# merge_df = pd.merge(metabolomics_merge_df,proteomics_merge_df,how = 'inner',on=['AIBL.ID', 'Collection','label'])
merge_df = pd.merge(metabolomics_merge_df,proteomics_df,how = 'inner',on=['AIBL.ID', 'Collection','Gender','label'])
merge_df = pd.merge(merge_df,check_amyloid[['AIBL.ID','Image.PET.Centiloid']],how = 'inner',on=['AIBL.ID'])
merge_df = merge_df.rename(columns={"Image.PET.Centiloid": "Centiloid"})
print(merge_df.shape)
print(merge_df.select_dtypes(include=['object', 'category']).columns)

(90, 1946)
Index(['Gender', 'label'], dtype='object')


#### Merge with AIBL

In [10]:
AIBL_raw = pd.read_excel('../raw_data/AIBL_2023.xlsx')
AIBL_raw['Demographic.YearMonthOfBirth']=AIBL_raw['Demographic.YearMonthOfBirth'].astype(str) + '01'
AIBL_raw['Demographic.YearMonthOfBirth'] = pd.to_datetime(AIBL_raw['Demographic.YearMonthOfBirth'], format='%Y%m%d')
AIBL_raw['Progress Summary.Date of NP assessment'] = pd.to_datetime(AIBL_raw['Progress Summary.Date of NP assessment'], format='%d/%m/%Y')
AIBL_raw['Age'] = (AIBL_raw['Progress Summary.Date of NP assessment'] - AIBL_raw['Demographic.YearMonthOfBirth']).dt.days // 365

In [11]:
AIBL_need = AIBL_raw.copy()
AIBL_need = AIBL_need.rename(columns = {'AIBL ID':'AIBL.ID'})
omics_merge_aibl_df = pd.merge(merge_df,AIBL_need,how = 'left',on = ['AIBL.ID', 'Collection'])
print("Dimension: ", omics_merge_aibl_df.shape)
# omics_merge_aibl_df.to_csv('../preprocessed_data/omics_merge_aibl.csv')

Dimension:  (90, 3345)


In [12]:
print(merge_df.shape)
aibl_demo_df = omics_merge_aibl_df[['AIBL.ID', 'Collection', 'Demographic.ApoE genotype','Demographic.Years of Education','Demographic.Years of Education Exact', 'Age', 'Demographic.Sex',
                                    'Demographic.Primary Occupation', 'Demographic.Retirement Status','Demographic.Current Marital Status', 'Demographic.Country of Birth', 'Demographic.Handedness', 'Vital Signs.Body Mass Index', 
                                    'Demographic.GDS','Neuropsych.MMSE', 'Neuropsych.CDR Sum of Boxes', 'Neuropsych.Composite.AIBLPACCscore']]
aibl_demo_df['APOE4'] = np.where(
aibl_demo_df['Demographic.ApoE genotype'].str.contains('E4', na=False),
    1,
    np.where(aibl_demo_df['Demographic.ApoE genotype'].isna(),np.nan,0)
)
omics_with_demo_df = pd.merge(merge_df, aibl_demo_df, on = ['AIBL.ID', 'Collection'], how='inner')
print(omics_with_demo_df.shape)
# handness one-hot encoding,
if "Demographic.Handedness" in omics_with_demo_df:                                                # one-hot handedness
    oh = pd.get_dummies(omics_with_demo_df["Demographic.Handedness"], prefix="Demographic.Handedness")
    omics_with_demo_df = pd.concat([omics_with_demo_df.drop(columns=["Demographic.Handedness"]), oh], axis=1)
print(omics_with_demo_df.shape)

(90, 1946)
(90, 1962)
(90, 1965)


In [1]:
### make a name list for further use
label_names = ['label', 'Centiloid']
keys = ['AIBL.ID', 'Collection']
proteomics_cols = proteomics_cols
metabolomics_cols = metabolomics_cols
demographic_cols_one_hot = ['APOE4', 'Demographic.Years of Education', 'Demographic.Years of Education Exact', 'Age',
                    'Demographic.Sex', 'Demographic.Primary Occupation', 'Demographic.Retirement Status',
                    'Demographic.Current Marital Status', 'Demographic.Country of Birth', 
                    "Demographic.Handedness_Left", "Demographic.Handedness_Right", "Demographic.Handedness_Ambidextrous",
                    'Vital Signs.Body Mass Index'] 
neuropsychological_cols = ['Demographic.GDS', 'Neuropsych.MMSE', 'Neuropsych.CDR Sum of Boxes', 'Neuropsych.Composite.AIBLPACCscore']

max_len = max(len(keys), len(label_names), len(demographic_cols_one_hot), 
              len(proteomics_cols), len(metabolomics_cols), len(neuropsychological_cols))

data = { 
    'keys': keys + [None] * (max_len - len(keys)),
    'label_names': label_names + [None] * (max_len - len(label_names)),
    'demographic_cols': demographic_cols_one_hot + [None] * (max_len - len(demographic_cols_one_hot)),
    'proteomics_cols': proteomics_cols + [None] * (max_len - len(proteomics_cols)),
    'metabolomics_cols': metabolomics_cols + [None] * (max_len - len(metabolomics_cols)),
    'neuropsychological_cols': neuropsychological_cols + [None] * (max_len - len(neuropsychological_cols))
}

# Create DataFrame and save
df = pd.DataFrame(data)
df.to_excel('../preprocessed_data/feature_names.xlsx', index=False)

print("File 'feature_names.xlsx' created successfully!")
print(f"\nShape: {df.shape}")

NameError: name 'proteomics_cols' is not defined

In [ ]:
# omics_with_demo_df[label_names]
# omics_with_demo_df[keys]
# omics_with_demo_df[proteomics_cols]
# omics_with_demo_df[metabolomics_cols]
# omics_with_demo_df[demographic_cols_one_hot]
# omics_with_demo_df[neuropsychological_cols]

,Demographic.GDS,Neuropsych.MMSE,Neuropsych.CDR Sum of Boxes,Neuropsych.Composite.AIBLPACCscore
0,0.0,30.0,0.0,0.868480
1,1.0,28.0,0.0,0.308529
2,0.0,28.0,0.0,0.500533
3,0.0,27.0,0.0,-0.945665
4,3.0,30.0,0.0,0.478234
...,...,...,...,...
85,1.0,30.0,0.0,0.650641
86,1.0,30.0,0.0,1.083644
87,0.0,30.0,0.0,0.798562
88,1.0,30.0,0.0,0.801368


In [14]:
missing_values = omics_with_demo_df.isnull().sum()
missing_info = {col: count for col, count in missing_values.items() if count > 0}
print(missing_info)
omics_with_demo_df.to_csv('../preprocessed_data/omics_aibl_demo.csv',index=False)

{'Demographic.ApoE genotype': 2, 'Demographic.Years of Education Exact': 6, 'Demographic.Primary Occupation': 2, 'Demographic.Retirement Status': 1, 'Demographic.Current Marital Status': 1, 'Demographic.Country of Birth': 1, 'Vital Signs.Body Mass Index': 10, 'Neuropsych.CDR Sum of Boxes': 1, 'Neuropsych.Composite.AIBLPACCscore': 2, 'APOE4': 2}


In [15]:
######## Go to R file (imputation.Rmd) to impute only demographic information